# MovieGPT training monitor

Plot the metrics written by `train_gpt2.py` without starting training or loading a model.
Run all cells again to refresh the graphs while training runs in another terminal.

Install dependencies with `python -m pip install -r requirements.txt -r requirements-notebook.txt`
and select your project's Python kernel. The default input is `log/log.txt`.
No training data is fabricated when the log is missing.

## Setup

Change `LOG_DIR` if the kernel starts outside the repository or you saved another run.
Smoothing is a trailing average of training records; raw loss stays visible.
Set `SAVE_PLOTS = True` to export PNGs to the run's `plots` folder.

In [ ]:
from pathlib import Path
import math
import warnings

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

LOG_DIR = Path("log")
SMOOTHING_WINDOW = 50
SAVE_PLOTS = False

plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 10})


## Read the latest logs

Each complete line is `step metric value`. Steps are optimizer steps. Invalid or
non-finite records are reported and skipped. An unfinished trailing line is ignored
until the next refresh. If a step/metric appears twice, the latest value wins.
Training auto-resumes matching local checkpoints; fresh runs require an unused directory. Resume preserves history through the
checkpoint and archives any abandoned tail. Copy the log directory before starting
an unrelated run if you want to keep its history.

In [ ]:
def read_metrics(path):
    names = ("train", "val", "perplexity", "comprehension_sanity")
    records = {name: {} for name in names}
    if not path.exists():
        print(f"No training log yet: {path.resolve()}. Start training, then rerun this notebook.")
        return records
    text = path.read_text()
    lines = text.splitlines()
    if text and not text.endswith("\n"):
        lines = lines[:-1]
    invalid = duplicates = 0
    for line in lines:
        fields = line.split()
        if len(fields) != 3:
            invalid += 1
            continue
        step_text, metric, value_text = fields
        if metric not in records:
            continue
        try:
            step, value = int(step_text), float(value_text)
            if step < 0 or not math.isfinite(value):
                raise ValueError
            if metric == "comprehension_sanity" and not 0 <= value <= 1:
                raise ValueError
            if metric in ("train", "val") and value < 0:
                raise ValueError
            if metric == "perplexity" and value < 1:
                raise ValueError
        except ValueError:
            invalid += 1
            continue
        duplicates += step in records[metric]
        records[metric][step] = value
    if invalid or duplicates:
        warnings.warn(f"Skipped {invalid} invalid records; replaced {duplicates} duplicates.")
    print(f"Source: {path.resolve()}")
    return records


def series(records, metric):
    pairs = sorted(records[metric].items())
    return (np.array([step for step, _ in pairs], dtype=int),
            np.array([value for _, value in pairs], dtype=float))


metrics = read_metrics(LOG_DIR / "log.txt")
# Older logs may contain validation loss without explicit perplexity entries.
for step, loss in metrics["val"].items():
    metrics["perplexity"].setdefault(step, math.exp(min(loss, 80)))

rows = ["| Metric | Records | Latest step | Latest value |",
        "| --- | ---: | ---: | ---: |"]
for metric, records in metrics.items():
    if records:
        step = max(records)
        rows.append(f"| {metric} | {len(records)} | {step} | {records[step]:.4f} |")
    else:
        rows.append(f"| {metric} | 0 | — | — |")
display(Markdown("\n".join(rows)))
if metrics["val"]:
    best_step = min(metrics["val"], key=metrics["val"].get)
    print(f"Lowest logged validation loss: {metrics['val'][best_step]:.4f} at step {best_step}.")


## Training and validation loss

Lower cross-entropy is better. Training loss is measured on changing training batches;
validation loss uses the same held-out window each evaluation. A sustained rise in
validation loss while training loss falls can indicate overfitting. The lowest logged
loss does not necessarily correspond to a saved checkpoint.

In [ ]:
fig_loss, ax = plt.subplots(figsize=(10, 4.5), layout="constrained")
train_steps, train_loss = series(metrics, "train")
val_steps, val_loss = series(metrics, "val")
if len(train_loss):
    ax.plot(train_steps, train_loss, color="#597aab", alpha=0.3, linewidth=1,
            label="Training (raw)")
    window = min(max(1, int(SMOOTHING_WINDOW)), len(train_loss))
    cumulative = np.concatenate(([0.0], np.cumsum(train_loss)))
    stops = np.arange(1, len(train_loss) + 1)
    starts = np.maximum(0, stops - window)
    smoothed = (cumulative[stops] - cumulative[starts]) / (stops - starts)
    ax.plot(train_steps, smoothed, color="#245d9b", linewidth=1.8,
            label=f"Training (trailing {window} records)")
if len(val_loss):
    ax.plot(val_steps, val_loss, "o-", color="#c56022", markersize=4,
            label="Validation")
if len(train_loss) or len(val_loss):
    ax.legend()
else:
    ax.text(0.5, 0.5, "No loss records yet", transform=ax.transAxes, ha="center")
ax.set(title="MovieGPT · next-token loss", xlabel="Optimizer step",
       ylabel="Cross-entropy (nats/token)")
ax.grid(alpha=0.2)
plt.show()


## Perplexity and comprehension

Perplexity is `exp(validation loss)` (exponent capped at 80 by the trainer), shown on a
logarithmic axis. Lower is better. It measures text prediction, not sentence quality.
The eight-question comprehension diagnostic is coarse: one answer changes the score
by 12.5 percentage points. Its chance reference is 1/3 for three choices; this is not
a calibrated benchmark. Validation covers a fixed 20-batch window, not the full corpus.

In [ ]:
fig_eval, axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
for ax, metric, title, ylabel, color in [
    (axes[0], "perplexity", "Held-out perplexity", "Perplexity (log scale)", "#c56022"),
    (axes[1], "comprehension_sanity", "Comprehension sanity check", "Accuracy", "#248478"),
]:
    steps, values = series(metrics, metric)
    if len(values):
        ax.plot(steps, values, "o-", color=color, markersize=4)
    else:
        ax.text(0.5, 0.5, "No evaluation records yet", transform=ax.transAxes, ha="center")
    ax.set(title=title, xlabel="Optimizer step", ylabel=ylabel)
    ax.grid(alpha=0.2)
axes[0].set_yscale("log")
axes[1].set_ylim(-0.03, 1.03)
axes[1].axhline(1 / 3, linestyle="--", color="gray", label="Chance (1/3)")
axes[1].legend()
plt.show()

if SAVE_PLOTS:
    output_dir = LOG_DIR / "plots"
    output_dir.mkdir(parents=True, exist_ok=True)
    fig_loss.savefig(output_dir / "loss.png", dpi=180)
    fig_eval.savefig(output_dir / "evaluation.png", dpi=180)
    print(f"Saved plots to {output_dir.resolve()}")


## Inspect generated dialogue

Check grammatical sentences, relevant replies, consistent facts, and repetition
alongside the curves. This preview shows the tail of `samples.txt`; open the full file
to compare checkpoints. The trainer appends samples across runs, so archive or clear
that file between runs when comparing it with a newly overwritten metrics log.

In [ ]:
samples_path = LOG_DIR / "samples.txt"
if samples_path.exists():
    # Read a bounded tail even for long training runs.
    with samples_path.open("rb") as handle:
        handle.seek(0, 2)
        size = handle.tell()
        handle.seek(max(0, size - 6000))
        tail = handle.read().decode("utf-8", errors="replace")
    print(("[Last 6,000 bytes]\n" if size > 6000 else "") + tail)
else:
    print("No samples yet. The trainer saves dialogue samples after step 0.")
